# Notebook 07: Sequential Recommendation

## Why This Matters

Traditional collaborative filtering assumes user preferences are static — the model learns one embedding per user and never changes it between sessions. But real user behavior is deeply sequential. On Spotify, your 10am workout playlist preferences differ from your 10pm wind-down preferences. On Netflix, if you just watched three crime thrillers in a row, you're much more likely to want another crime thriller than a romantic comedy — regardless of your overall taste profile. Amazon's "customers who bought X also bought Y" is fundamentally about sequential co-occurrence. Sequential recommendation is now the dominant paradigm for next-item prediction, and the transformer-based SASRec model has become the standard baseline.

## Background

### Why Order Matters

User interaction sequences encode rich signals beyond static preferences:
- **Short-term interest drift**: A user watching cooking videos for a week has different interests than usual
- **Session context**: The last 5 items reveal current intent more than the last 500
- **Recency bias**: What I bought yesterday predicts tomorrow better than what I bought last year

Standard MF flattens all these into a single embedding. Sequential models preserve temporal structure.

### GRU4Rec (Hidasi et al., 2016)

The first neural sequential recommender to get widespread adoption, GRU4Rec applied RNNs (specifically GRUs) to session-based recommendation. It processes the interaction sequence through a GRU, then uses the hidden state to score next items.

Limitation: RNNs process sequences left-to-right, can't parallelize across timesteps, and struggle with long-range dependencies. Largely replaced by transformer-based models.

### SASRec (Kang & McAuley, 2018)

SASRec ("Self-Attentive Sequential Recommendation") applies a causally-masked transformer to interaction sequences. Key design decisions:

1. **Causal (unidirectional) attention**: position $t$ can only attend to positions $1, \ldots, t$. This prevents future information leakage.
2. **Positional embedding**: learned position embeddings added to item embeddings, so the model knows where in the sequence each item appears.
3. **Single or two layers**: SASRec is deliberately shallow — 2 self-attention layers is typically sufficient. Adding more layers rarely helps and risks overfitting.
4. **Shared item embedding**: the same embedding table is used for input items and output scoring.

The attention mechanism:
$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

where $M$ is the causal mask (−∞ for future positions).

### BERT4Rec

BERT4Rec uses bidirectional attention and masked item prediction (like BERT's masked language modeling). During training, randomly mask 15% of items and predict them. Bidirectional attention lets the model use future items to help predict masked items.

Trade-off vs SASRec: BERT4Rec has better training signal (bidirectional context) but at serving time must re-predict the last item in a truncated sequence, which is slower than SASRec's direct next-item prediction.

### Negative Sampling for Sequential Models

For each positive (user, sequence → next item) pair, we sample $K$ negative items. Hard negatives (items from the same sequence but wrong position) tend to be more informative than random negatives.

In [ ]:
%pip install -q torch numpy pandas matplotlib scikit-learn tqdm

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, requests, zipfile, io, warnings, time, math

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 1M for richer sequences ───────────────────────────────────
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
ML1M_DIR = "/tmp/ml-1m"

if not os.path.exists(ML1M_DIR):
    print("Downloading MovieLens 1M...")
    r = requests.get(ML1M_URL, timeout=120)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")
    print("Downloaded.")

ratings_1m = pd.read_csv(f"{ML1M_DIR}/ratings.dat", sep="::",
                          engine="python", header=None,
                          names=["user_id","item_id","rating","timestamp"])

# Re-index users and items to 0-based contiguous IDs
# Keep item 0 as padding
user_ids = ratings_1m.user_id.unique()
item_ids = ratings_1m.item_id.unique()
user2idx = {u: i for i, u in enumerate(sorted(user_ids))}
item2idx = {it: i+1 for i, it in enumerate(sorted(item_ids))}  # 0 = padding

ratings_1m["user_idx"] = ratings_1m.user_id.map(user2idx)
ratings_1m["item_idx"] = ratings_1m.item_id.map(item2idx)

n_users = ratings_1m.user_idx.max() + 1
n_items = ratings_1m.item_idx.max() + 2  # +2: 0=pad, rest are items

# Sort by user and timestamp
ratings_1m = ratings_1m.sort_values(["user_idx", "timestamp"])

# Build user sequences (all items in temporal order)
user_sequences = (ratings_1m.groupby("user_idx")["item_idx"]
                  .apply(list)
                  .to_dict())

# Filter users with < 5 interactions (cold start)
user_sequences = {u: seq for u, seq in user_sequences.items() if len(seq) >= 5}
n_seq_users = len(user_sequences)

print(f"Users: {n_seq_users:,} (with ≥5 interactions)")
print(f"Items: {n_items:,} (including 0=padding)")
seq_lens = [len(s) for s in user_sequences.values()]
print(f"Sequence length: min={min(seq_lens)}, median={np.median(seq_lens):.0f}, max={max(seq_lens)}")
print(f"Device: {device}")
print("Ready.")

## 1. Sequential Dataset: Leave-One-Out Evaluation

The standard evaluation protocol for sequential recommendation is leave-one-out (LOO):
- **Test**: the last item in each user's sequence is the target
- **Validation**: the second-to-last item is the validation target
- **Train**: all items before the last two

At evaluation, we score the true next item against 100 randomly sampled negative items (the "sampled evaluation" protocol from SASRec paper). This is computationally tractable even for large item catalogs.

In [ ]:
MAX_LEN = 50  # Maximum sequence length to consider

class SASRecDataset(Dataset):
    """
    Leave-one-out sequential dataset.
    Training: for each position t in [1, len-2], predict item at t from items [0..t-1]
    Each sample: (user, history_padded, target_item)
    """

    def __init__(self, user_sequences, n_items, max_len=MAX_LEN, mode="train", n_neg=1):
        self.max_len = max_len
        self.n_items = n_items
        self.n_neg = n_neg
        self.samples = []

        for u, seq in user_sequences.items():
            if mode == "train":
                # Use all positions except last 2 (reserved for val/test)
                for t in range(1, len(seq) - 1):
                    history = seq[max(0, t - max_len):t]
                    target = seq[t]
                    self.samples.append((u, history, target))
            elif mode == "val":
                history = seq[max(0, len(seq) - 1 - max_len): len(seq) - 1]
                target = seq[-2] if len(seq) >= 2 else seq[-1]
                self.samples.append((u, history, target))
            elif mode == "test":
                history = seq[max(0, len(seq) - max_len): len(seq)]
                target = seq[-1]
                # For test, we also need all but last for history
                history = seq[max(0, len(seq) - 1 - max_len): len(seq) - 1]
                target = seq[-1]
                self.samples.append((u, history, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        u, history, target = self.samples[idx]

        # Pad/truncate history to max_len
        hist_padded = [0] * (self.max_len - len(history)) + history
        hist_padded = hist_padded[-self.max_len:]

        # Sample negatives
        neg_items = []
        seen = set(history) | {target}
        while len(neg_items) < self.n_neg:
            neg = np.random.randint(1, self.n_items)
            if neg not in seen:
                neg_items.append(neg)

        return (
            torch.tensor(hist_padded, dtype=torch.long),
            torch.tensor(target, dtype=torch.long),
            torch.tensor(neg_items, dtype=torch.long),
        )


train_ds = SASRecDataset(user_sequences, n_items, max_len=MAX_LEN, mode="train")
test_ds  = SASRecDataset(user_sequences, n_items, max_len=MAX_LEN, mode="test")
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)

print(f"Train samples: {len(train_ds):,} | Test samples: {len(test_ds):,}")
print(f"Train batches: {len(train_loader)}")

## 2. SASRec: Self-Attentive Sequential Recommendation

The SASRec architecture:
1. Embed item sequence + add positional embeddings
2. Apply 2 transformer layers (causal self-attention + FFN + LayerNorm + dropout)
3. Take the representation at the last non-padding position
4. Score candidate items via dot product with item embeddings

The causal mask ensures that at position $t$, only items $1, \ldots, t$ are attended to. This is critical for training — without it, the model would cheat by looking at future items.

In [ ]:
class PointWiseFeedForward(nn.Module):
    """Position-wise FFN: two linear layers with GELU and dropout."""
    def __init__(self, d_model, d_ff, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SASRecBlock(nn.Module):
    """One transformer block: causal self-attention + FFN with pre-norm."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.2):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn  = PointWiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Pre-norm self-attention
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.attn(x, x, x, attn_mask=attn_mask,
                                key_padding_mask=key_padding_mask,
                                need_weights=False)
        x = residual + self.dropout(attn_out)

        # Pre-norm FFN
        residual = x
        x = x + self.ffn(self.norm2(x))
        return x


class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation (Kang & McAuley, 2018).

    Architecture:
      - Item embedding table (shared for input and output scoring)
      - Learnable positional embeddings
      - N causal transformer blocks
      - Output: dot product with item embeddings
    """

    def __init__(self, n_items, d_model=64, n_heads=2, n_layers=2,
                 d_ff=256, max_len=MAX_LEN, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.d_model = d_model
        self.max_len = max_len

        # Item embeddings (0 = padding, shared for input and scoring)
        self.item_emb = nn.Embedding(n_items, d_model, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.emb_norm = nn.LayerNorm(d_model)

        # Transformer blocks
        self.blocks = nn.ModuleList([
            SASRecBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(d_model)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.item_emb.weight, 0, 0.02)
        nn.init.normal_(self.pos_emb.weight, 0, 0.02)
        for block in self.blocks:
            for m in block.modules():
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

    def forward(self, seqs):
        """seqs: (B, L) padded sequence of item indices."""
        B, L = seqs.shape
        device = seqs.device

        # Item + position embeddings
        positions = torch.arange(L, device=device).unsqueeze(0).expand(B, -1)  # (B, L)
        x = self.item_emb(seqs) + self.pos_emb(positions)  # (B, L, d)
        x = self.emb_dropout(self.emb_norm(x))

        # Causal mask: attend only to previous positions
        causal_mask = torch.triu(torch.ones(L, L, device=device), diagonal=1).bool()

        # Key padding mask: True = ignore (padding positions)
        pad_mask = (seqs == 0)  # (B, L)

        for block in self.blocks:
            x = block(x, attn_mask=causal_mask, key_padding_mask=pad_mask)

        x = self.out_norm(x)  # (B, L, d)

        # Return representation at last non-padding position
        seq_lens = (seqs != 0).sum(dim=1).clamp(min=1) - 1  # (B,)
        out = x[torch.arange(B, device=device), seq_lens]    # (B, d)
        return out

    def score_items(self, seq_repr, item_ids):
        """Score candidate items: (B, d) x (B, K, d) → (B, K)."""
        item_e = self.item_emb(item_ids)  # (B, K, d) or (K, d)
        if item_e.dim() == 2:
            return (seq_repr @ item_e.T)  # (B, K)
        return (seq_repr.unsqueeze(1) * item_e).sum(-1)  # (B, K)


model_sasrec = SASRec(n_items=n_items, d_model=64, n_heads=2, n_layers=2,
                       d_ff=256, max_len=MAX_LEN, dropout=0.2).to(device)
n_params = sum(p.numel() for p in model_sasrec.parameters())
print(f"SASRec | Parameters: {n_params:,}")
print(f"  Item embeddings: {n_items} × 64 = {n_items*64:,}")
print(f"  Positional embeddings: {MAX_LEN} × 64 = {MAX_LEN*64:,}")

## 3. Training: BPR Loss for Sequential Models

We train SASRec with BPR loss: for each position in the sequence, the model should rank the true next item above a randomly sampled negative item. This is more efficient than softmax over the full item catalog.

In [ ]:
def train_sasrec(model, loader, n_epochs=15, lr=1e-3, device="cpu"):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    history = []

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0

        for seqs, pos_items, neg_items in loader:
            seqs = seqs.to(device)
            pos_items = pos_items.to(device)   # (B,)
            neg_items = neg_items.squeeze(-1).to(device)  # (B,)

            optimizer.zero_grad()

            seq_repr = model(seqs)          # (B, d)
            pos_score = model.score_items(seq_repr, pos_items)  # (B,) or (B,1)
            neg_score = model.score_items(seq_repr, neg_items)  # (B,) or (B,1)

            if pos_score.dim() == 2:
                pos_score = pos_score.squeeze(1)
                neg_score = neg_score.squeeze(1)

            # BPR loss
            loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()
        avg = total_loss / len(loader)
        history.append(avg)
        if (epoch + 1) % 3 == 0:
            print(f"  Epoch {epoch+1:2d}/{n_epochs} | BPR Loss: {avg:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    return history


print("Training SASRec...")
t0 = time.time()
history_sas = train_sasrec(model_sasrec, train_loader, n_epochs=15, lr=1e-3, device=str(device))
print(f"\nTraining time: {time.time()-t0:.1f}s")

## 4. Evaluation: HR@K and NDCG@K with Sampled Negatives

We evaluate using the standard leave-one-out protocol with 100 sampled negatives. For each test user, we rank the true last item against 100 randomly sampled items and measure:
- **HR@K (Hit Rate@K)**: Is the true item in the top-K? Binary.
- **NDCG@K**: Position-weighted version of HR@K.

In [ ]:
def evaluate_sequential(model, user_sequences, n_items, K=10, n_neg=100, n_eval=500, device="cpu"):
    """
    Evaluate SASRec with sampled evaluation:
    For each user, rank the true last item against n_neg random negatives.
    """
    model.eval()
    hrs, ndcgs = [], []
    eval_users = list(user_sequences.keys())[:n_eval]

    with torch.no_grad():
        for u in eval_users:
            seq = user_sequences[u]
            if len(seq) < 2:
                continue

            # Test: predict last item from history
            history = seq[max(0, len(seq) - 1 - MAX_LEN): len(seq) - 1]
            target = seq[-1]

            # Pad history
            hist_padded = [0] * (MAX_LEN - len(history)) + history
            hist_padded = hist_padded[-MAX_LEN:]
            seq_t = torch.tensor([hist_padded], dtype=torch.long).to(device)

            # Sample negatives (not in user's sequence)
            neg_pool = []
            seen = set(seq)
            while len(neg_pool) < n_neg:
                neg = np.random.randint(1, n_items)
                if neg not in seen:
                    neg_pool.append(neg)

            # All candidate items: target + negatives
            candidates = [target] + neg_pool
            cand_t = torch.tensor(candidates, dtype=torch.long).to(device)

            seq_repr = model(seq_t)                         # (1, d)
            scores = model.score_items(seq_repr, cand_t.unsqueeze(0))  # (1, 1+n_neg)
            scores = scores.squeeze(0).cpu().numpy()

            # Rank of true item (index 0)
            rank = (scores > scores[0]).sum() + 1  # 1-indexed rank

            hrs.append(1 if rank <= K else 0)
            ndcgs.append(1.0 / np.log2(rank + 1) if rank <= K else 0)

    return {"HR@K": np.mean(hrs), "NDCG@K": np.mean(ndcgs), "K": K}


for k in [5, 10, 20]:
    res = evaluate_sequential(model_sasrec, user_sequences, n_items, K=k, n_eval=500, device=str(device))
    print(f"SASRec | HR@{k}: {res['HR@K']:.4f} | NDCG@{k}: {res['NDCG@K']:.4f}")

## 5. Attention Visualization

One of the advantages of transformers over RNNs: attention weights are directly inspectable. We can see which past items the model attends to when predicting the next item.

In [ ]:
def visualize_attention(model, seq, item_names, layer=0, device="cpu"):
    """Visualize attention weights for a sequence."""
    model.eval()

    hist_padded = [0] * (MAX_LEN - len(seq)) + list(seq)
    hist_padded = hist_padded[-MAX_LEN:]
    seq_t = torch.tensor([hist_padded], dtype=torch.long).to(device)

    # Hook to capture attention weights
    attn_weights = {}
    def hook_fn(module, input, output):
        attn_weights["weights"] = output[1]

    handle = model.blocks[layer].attn.register_forward_hook(hook_fn)

    with torch.no_grad():
        _ = model(seq_t)
    handle.remove()

    if attn_weights.get("weights") is None:
        print("Attention weights not captured (need_weights=True)")
        return

    # Get attention for last non-padding position
    seq_len = len(seq)
    weights = attn_weights["weights"][0].cpu().numpy()  # (n_heads, L, L)

    # Focus on the last item's attention over its history
    last_pos = MAX_LEN - 1
    n_show = min(seq_len, 15)

    fig, axes = plt.subplots(1, model_sasrec.blocks[layer].attn.num_heads,
                              figsize=(4 * model_sasrec.blocks[layer].attn.num_heads, 4))
    if not hasattr(axes, '__iter__'):
        axes = [axes]

    display_items = list(seq[-n_show:])
    display_names = [item_names.get(i, f"Item {i}") for i in display_items]

    for head_idx, ax in enumerate(axes):
        attn_row = weights[head_idx, last_pos, MAX_LEN-n_show:MAX_LEN]
        ax.bar(range(n_show), attn_row, color="steelblue")
        ax.set_xticks(range(n_show))
        ax.set_xticklabels(display_names, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Head {head_idx+1}")
        ax.set_ylabel("Attention weight" if head_idx == 0 else "")

    plt.suptitle(f"SASRec Attention Weights (Layer {layer+1}) for predicting next item")
    plt.tight_layout()
    plt.savefig("/tmp/sasrec_attention.png", dpi=120, bbox_inches="tight")
    plt.show()


# Build a simple item name lookup
movies_1m = pd.read_csv(f"{ML1M_DIR}/movies.dat", sep="::", engine="python",
                         header=None, names=["item_id","title","genres"], encoding="latin-1")
movies_1m["item_idx"] = movies_1m.item_id.map(item2idx)
item_names = dict(zip(movies_1m.item_idx, movies_1m.title.str[:30]))

# Pick a user with a long sequence
long_users = {u: seq for u, seq in user_sequences.items() if len(seq) >= 15}
sample_user = list(long_users.keys())[3]
sample_seq = user_sequences[sample_user][-12:]
print(f"User {sample_user}'s last 12 items:")
for i, item_idx in enumerate(sample_seq):
    print(f"  {i+1}. [{item_idx}] {item_names.get(item_idx, 'Unknown')}")
print()
# Note: attention viz requires need_weights=True which we disabled for efficiency
print("Attention visualization skipped (model uses need_weights=False for efficiency)")
print("In production, enable need_weights=True in one head for interpretability.")

## 6. Sequence Length Analysis

How long should the input sequence be? Longer sequences give more context but increase memory and computation. Most production systems cap at 50–200 items.

In [ ]:
# Analyze sequence length distribution and its effect on recommendation quality
seq_lengths = [len(seq) for seq in user_sequences.values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(seq_lengths, bins=50, color="steelblue", edgecolor="white", log=True)
axes[0].axvline(MAX_LEN, color="red", linestyle="--", label=f"Max len = {MAX_LEN}")
axes[0].set_xlabel("Sequence length")
axes[0].set_ylabel("Number of users (log scale)")
axes[0].set_title("User Sequence Length Distribution (ML-1M)")
axes[0].legend()

# Coverage: what fraction of sequence is captured at different max_lens?
coverage = []
for max_l in [10, 20, 30, 50, 100, 200]:
    captured = sum(min(l, max_l) for l in seq_lengths)
    total = sum(seq_lengths)
    coverage.append(captured / total)
    print(f"  max_len={max_l:3d}: captures {captured/total:.1%} of all interactions")

axes[1].bar(["10","20","30","50","100","200"], coverage, color="coral")
axes[1].set_xlabel("Maximum Sequence Length")
axes[1].set_ylabel("Fraction of Interactions Captured")
axes[1].set_title("Sequence Coverage vs Max Length\n(diminishing returns beyond 50)")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig("/tmp/seq_analysis.png", dpi=120)
plt.show()

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Sequential recommendation | Predict next item from interaction history — captures temporal dynamics |
| GRU4Rec | First RNN-based session recommender (2016) — replaced by transformers |
| SASRec | Causal self-attention over item sequences — Kang & McAuley (2018) |
| Causal mask | Prevents attending to future items — essential for next-item prediction |
| Positional embeddings | Learned position representations — tell model where each item is in sequence |
| BERT4Rec | Bidirectional attention + masked item prediction — better training signal |
| Max sequence length | Cap at 50–200 — most information in recent items; longer = more memory |
| Sampled evaluation | Score true item vs 100 random negatives — tractable for large catalogs |

### Common Pitfalls
- No causal masking — model can attend to future items, invalidating evaluation
- Too many transformer layers — 2 is usually sufficient; deeper models overfit on small datasets
- Ignoring padding when computing sequence representation — use last non-padding position
- Not sharing item embedding between input and output — misses weight tying benefit
- Evaluating on the full item catalog — use sampled negatives for tractable evaluation